# ED5J990H5VAZT

Packages

In [ ]:
# Imports
import os
from pathlib import Path
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())
from imports import *
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_3_x= return_dir()

# Settings
notebook_settings() 

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
loc_id = 'ED5J990H5VAZT'
df_uncleaned = load_single_restaurant(loc_id)

In [ ]:
print(df_uncleaned.query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")').query('item_name.str.contains("Bacon") or item_modifications.str.contains("Bacon")')['item_quantity'].sum())
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")').query('item_name.str.contains("Bacon") or item_modifications.str.contains("Bacon")')['item_quantity'].sum())

### Dish Consolidation

In [ ]:
example = [('Griffith Street','Vegan Bacon|Thrilling Foods|No Bacon|Vegan Sausage|Veggie Sausage|No Sausage|Field Roast|No Meat','Nonmeat Griffith Street'),
 ('Griffith Street','Bacon|Sausage|Turkey|Meat','Meat Griffith Street'),
 ('Griffith Street','','Nonmeat Griffith Street'), # Default meat based on menu
 
 ('Nonmeat Griffith Street','Good Planet|Just Egg|Vegan Cream','Nonegg Griffith Street'),
 ('Nonmeat Griffith Street','Cheddar|Cheese|Egg|Havarti','Vegetarian Griffith Street'),
 ('Nonmeat Griffith Street','','Vegetarian Griffith Street')] # Default cheese based on menu

drink_categories = ["Coffee & Tea","Dairy Drink","Alcohol","Soda","Water","Juice","Sports & Health Drink"]


# Item names to consolidate
dish_names = {"Griffith Street" : ["Gs", "Gs Meat"], # Vegan Griffith Street 
              "Awkward Aardvark" : ["Aa/No Meat"], 
              "Egg & Cheddar" : ["Ec", "Ec Tomato", "Egg Cheddar", "Egg Cheese"],  # "Vegan Egg & Cheddar", "Vegan Egg & Cheese"
              "Egg Hummus Pesto" : ["Ehp", "Ehp Tomato", "Ehp Meat", "Egg Hummus-Pesto"], # "Vegan Egg Hummus Pesto"
              "Egg Meat Cheese" : ["Ec Meat", "3. Egg Meat Cheese"],
              "Egg Meat" : ["Egg & Bacon"],
              "Egg": ["Scrambled Eggs"], # "Egg Sandwich"
              "Loose-Leaf Tea" : ["Loose-Leaf", "Loose Leaf"],
              "Veggie Sandwich" : ["Veggie Sandwich", "Veggie Sandwich With Side Salad"],}

MODS_VEGAN_MEAT = 'Vegan Bacon|Thrilling Foods|No Bacon|Vegan Sausage|Veggie Sausage|No Sausage|Field Roast|No Meat'
MODS_MEAT = 'Bacon|Sausage|Turkey|Meat'
MODS_VEGAN_DAIRY_EGG = 'Good Planet|Just Egg|Vegan Cream|Vegan Egg'
MODS_DAIRY_EGG = 'Cheddar|Cheese|Egg|Havarti'
MODS_EXPLICIT_VEGAN = 'Vegan'
MODS_NONE = ''

# Define base dishes, their final default state, and if step 2 has an explicit 'Vegan' mod rule
# Format: (base_name, final_default_state, has_explicit_nonmeat_vegan_rule)
DISH_CONFIG = [
    ('Veggie Sandwich', 'Vegetarian', True),
    ('Toby Toast', 'Vegan', False),
    ('Careless Whisper', 'Vegan', False),
    ('Garden Home', 'Vegan', False),
    ('Avocado Toast', 'Vegan', False),
    ('Griffith Street', 'Vegetarian', True),
    ('Awkward Aardvark', 'Meat', True),
    ('Egg & Cheddar', 'Vegetarian', True),
    ('Build Your Own', 'Meat', True),
    ('Yeti Sandwich', 'Meat', True),
    ('Hashy', 'Meat', True),
    ('The Craven', 'Meat', True),
    ('Yeli Sandwich', 'Meat', True), # Assuming distinct item
    ('One Punch', 'Vegetarian', True),
    ('The Dook', 'Vegetarian', True),
    ('Egg Meat Cheese', 'Meat', True),
    ('The Paul', 'Meat', True),
    ('Fullmetal Alchemist', 'Meat', True),
]

# --- Generate Rules ---

modification_rules_step1 = []
modification_rules_step2 = []

for base_name, final_default_state, has_explicit_vegan_rule in DISH_CONFIG:
    # --- Step 1 Rules (Base -> Intermediate) ---
    intermediate_name = f"Nonmeat {base_name}"
    meat_intermediate_name = f"Meat {base_name}" # Although often overridden, needed for rule structure

    modification_rules_step1.extend([
        (base_name, MODS_VEGAN_MEAT, intermediate_name),
        (base_name, MODS_MEAT, meat_intermediate_name),
        (base_name, MODS_NONE, intermediate_name), # Default to Nonmeat first
    ])

    # --- Step 2 Rules (Intermediate -> Final) ---
    vegan_final_name = f"Vegan {base_name}"
    vegetarian_final_name = f"Vegetarian {base_name}"
    meat_final_name = f"Meat {base_name}"

    # Determine the final default name based on the configured state
    if final_default_state == 'Vegan':
        default_final_name = vegan_final_name
    elif final_default_state == 'Vegetarian':
        default_final_name = vegetarian_final_name
    else: # Assumed 'Meat'
        default_final_name = meat_final_name

    # Add rules targeting the intermediate 'Nonmeat...' name
    step2_rules_for_dish = [
        (intermediate_name, MODS_VEGAN_DAIRY_EGG, vegan_final_name),
        (intermediate_name, MODS_DAIRY_EGG, vegetarian_final_name),
    ]

    # Add the explicit 'Vegan' mod rule only if configured
    if has_explicit_vegan_rule:
        step2_rules_for_dish.append((intermediate_name, MODS_EXPLICIT_VEGAN, vegan_final_name))

    # Add the final default rule
    step2_rules_for_dish.append((intermediate_name, MODS_NONE, default_final_name))

    modification_rules_step2.extend(step2_rules_for_dish)

no_menu_yes_sales = ["Yeli Sandwich", "Egg Hummus Pesto", "The Jesse Sandwich"]

yes_menu_entrees = ["Hashy","The Craven",
                    "Awkward Aardvark", "Anam A Nom", 
                    "Griffith Street", "Egg & Cheddar", 
                    "The Paul", "Yeti Sandwich", 
                    "Fullmetal Alchemist", "One Punch", 
                    "Careless Whisper", "Toby Toast", 
                    "Egg Meat Cheese", "Egg Meat", 
                    "Egg", "Egg & Cheddar",
                    "Halsey Street", "Tab",
                    "The Dook", "Garden Home",
                    "Avobagel", "Bagel"]

# What is:
# Sandwich - Breakfast
# Flavor
# Ex. Flavor
# Kind
# Wow
# Pure

# Probably change
# Yeli Sandwich to Yeti Sandwich

vegan = [
    'Vegan Veggie Sandwich',
    'Vegan Toby Toast',
    'Vegan Careless Whisper',
    'Vegan Garden Home',
    'Vegan Avocado Toast',       # Added from rules
    'Vegan Griffith Street',     # Added from rules
    'Vegan Awkward Aardvark',  # Added from rules
    'Vegan Egg & Cheddar',       # Added from rules
    'Vegan Build Your Own',      # Added from rules
    'Vegan Yeti Sandwich',       # Added from rules
    'Vegan Hashy',               # Added from rules
    'Vegan The Craven',          # Added from rules
    'Vegan Yeli Sandwich',       # Added from rules
    'Vegan One Punch',           # Added from rules
    'Vegan The Dook',            # Added from rules
    'Vegan Egg Meat Cheese',     # Added from rules
    'Vegan The Paul',            # Added from rules
    'Vegan Fullmetal Alchemist'  # Added from rules
]

vegetarian = [
    'Vegetarian Veggie Sandwich',
    'Vegetarian Toby Toast',
    'Vegetarian Careless Whisper',
    'Vegetarian Garden Home',
    'Vegetarian Avocado Toast',    # Added from rules
    'Vegetarian Griffith Street',  # Added from rules
    'Vegetarian Awkward Aardvark',# Added from rules
    'Vegetarian Egg & Cheddar',    # Added from rules
    'Vegetarian Build Your Own',   # Added from rules
    'Vegetarian Yeti Sandwich',    # Added from rules
    'Vegetarian Hashy',            # Added from rules
    'Vegetarian The Craven',       # Added from rules
    'Vegetarian Yeli Sandwich',    # Added from rules
    'Vegetarian One Punch',        # Added from rules
    'Vegetarian The Dook',         # Added from rules
    'Vegetarian Egg Meat Cheese',  # Added from rules
    'Vegetarian The Paul',         # Added from rules
    'Vegetarian Fullmetal Alchemist',# Added from rules
    # Original items:
    'Pumpkin Bread',
    'Pandan Cookie',
    'Bread | Carrot Zucchini',
    'Loaf | Carrot Zucchini',
    'Cookie',
    'Bagel'
]

meat = [
    'Meat Veggie Sandwich',
    'Meat Toby Toast',
    'Meat Careless Whisper',
    'Meat Garden Home',
    'Meat Avocado Toast',        # Added from rules
    'Meat Griffith Street',      # Added from rules
    'Meat Awkward Aardvark',   # Added from rules
    'Meat Egg & Cheddar',        # Added from rules
    'Meat Build Your Own',       # Added from rules
    'Meat Yeti Sandwich',        # Added from rules
    'Meat Hashy',                # Added from rules
    'Meat The Craven',           # Added from rules
    'Meat Yeli Sandwich',        # Added from rules
    'Meat One Punch',            # Added from rules
    'Meat The Dook',             # Added from rules
    'Meat Egg Meat Cheese',      # Added from rules
    'Meat The Paul',             # Added from rules
    'Meat Fullmetal Alchemist'   # Added from rules
]
alcoholic_drinks = [] 
non_alcoholic_drinks = ["Matcha","Loose-Leaf Tea","Signature","Flavor"]
rare = []
unknown = []

merch = [
    "Whole Beans",
    "Signature", # Assuming this refers to a signature item that could be either, but placing in merch as it's ambiguous often used for non-food branding too. Re-evaluate if context suggests otherwise.
    "Gift Card",
    "Sticker",
    "Card",
    "Easy Sunday Club",
    "Earrings",
    "Mask - Kelly",
    "Seewhyzhang",
    "Little Gold Fox Design",
    "Pin",
    "Ezen Design",
    "Cathy Zhang",
    "Poppy", # Likely a design based on "Bookmark - Poppy Spider"
    "Christa Pierce",
    "Stubby",
    "Sloane Darling Art",
    "Hazelhoff",
    "Enamel Co",
    "Bottom", # Likely POS modifier
    "Top", # Likely POS modifier
    "Sasquatch",
    "Noteworthy",
    "Mike York",
    "Masks",
    "David Hazelhoff",
    "Destination Oregon",
    "Hank Knit - Mask $10",
    "Nickel Art Studio",
    "Wooden Greeting Card",
    "Squishable",
    "Fun Club",
    "Graphic Heart",
    "Joco",
    "Sticker - Name, Waves",
    "Corgi Mouse Pad",
    "Urban Retrospective",
    "Middle", # Likely POS modifier
    "The Bower Studio",
    "Corgi Socks",
    "Basket",
    "Imagination Spot",
    "Spoon/Spatula",
    "Hoodie",
    "John Skewes",
    "Sticker - Tanuki, Black",
    "Sticker - Bear Hug",
    "Oregon Puzzle 4-Pc. Coaster Set With Case",
    "Candle",
    "Sticker - Logo, Black",
    "Peter Pauper Press",
    "Susie Chriswisser",
    "Chemex - Filters",
    "Sticker - Tanuki, Rona",
    "Sticker Bundle",
    "Print - Large",
    "Tea Towel",
    "Sticker - Logo, Red",
    "Caitlin Keegan",
    "Notebook",
    "Things Are What You Make Of Them",
    "Happiness", # Abstract theme, likely merch
    "Adam J. Kurtz",
    "Dishcloth",
    "Illuminated Tarot",
    "Enamel Pin",
    "Baltique", # Utensil brand likely
    "Create Your Own Calm",
    "Tattoo Tarot",
    "Earthwell", # Drinkware brand likely
    "A Life Of Gratitude",
    "Ginger Anne",
    "Salt Vial Set", # The set itself is merch
    "Tomoko Alfonso",
    "Start Where You Are",
    "Chemex", # Brewer
    "Page At A Time",
    "Forlife", # Teaware/drinkware brand likely
    "Good Luck Cat (Cutout) Holographic Eyes Sticker",
    "Mixed Feelings",
    "Basic Witches", # Book/theme likely
    "Sticker - Vote",
    "Blanket",
    "Growth", # Abstract theme, likely merch
    "Choose Hope, Take Action",
    "Work/Life Balance",
    "To-Go Box",
    "Jennifer Joy",
    "Sticker - Daruma",
    "The Girls - Enamel Pin",
    "Card - Happy Mothers Day Ducks",
    "Card - Fuck Yeah Fox",
    "Leathers",
    "Sticker - Tanuki, Red",
    "Bill Perkins",
    "Sticker - Bee",
    "Sticker - My Heart Oregon",
    "Single Greeting Card Elephant Yellow (W)",
    "Earrings - Butterfly Wings",
    "Susie", # Likely Artist/Brand name
    "Card - Hello From Portland",
    "Postcard Set - Portland Bridges",
    "Hand Knit - Coffee Cuffs",
    "Larry Gets Lost In Portland", # Book
    "Ellen Injerd",
    "Katie Stanley",
    "Tattoo Tarot Journal",
    "Hand Kint", # Typo for Hand Knit?
    "Abigail Terry",
    "Good Luck Black Cat Enamel Pin",
    "My Neighbor Totoro Die Cut Lunch Bag - Gray",
    "Illuminated Playing Cards",
    "Card - Go Shorty",
    "Portland Small Tray",
    "Tote",
    "X16 - Portland Bridges", # Identifier/Print
    "Card - Thank You",
    "‰∫Îâπ¥ Zip Hoodie - Mustard", # Hoodie
    "Am I Overthinking This", # Book/Journal
    "Sticker - Name, Black",
    "Undercover Corgi In Avocado (7\")", # Plush toy
    "Good Luck Cat Journal",
    "Made Out Of Stars", # Book/Theme
    "Card - Sending Good Vibes",
    "Salt Tin", # Container
    "Nowhere Land", # Art/Theme
    "Mala Bracelets By Abby",
    "Tea Towel - Oregon",
    "Crowned Rabbit", # Brand/Art
    "Card - I Miss Your Face",
    "The Tattoo Coloring Book",
    "Sticker - Oregon",
    "Oregon Board", # Cutting/Serving board
    "Counting With Barefoot Critters", # Book
    "Scales Ki-Shirt",
    "Emily Windfield Martin", # Artist/Author
    "Larry Loves Portland", # Book
    "Mala By Abby",
    "Portland Abc", # Book/Theme
    "Card - Happy Bday",
    "I: The Girls Notebook",
    "Sticker - Life Is An Adventure",
    "Tomorrow I‚Äôll Be Kind", # Book
    "Lori Roberts",
    "Pin - Fuck Yeah Fox",
    "Clever", # Brand/Theme
    "Robyn Nicole",
    "Embroidered Rose Sweater",
    "Card - Rad Woman With Plants",
    "Sticker - Van",
    "Pin - Bee",
    "Backroom Reservation", # Service/Fee
    "Leather",
    "Mug - Ceramic",
    "Card - Giraffe Family",
    "Reading Fox", # Design/Theme
    "Ki-Hoodie (Black Camo)",
    "Large Notebook Elephants Light Green (W)",
    "Ki-Shirt (Black)",
    "Bear Hug Sticker",
    "Card - Daddy Shark",
    "Card - Muttflix",
    "Card - You Are Magic",
    "X", # Identifier
    "Oregon State Stamp Series Salt Box", # Container
    "Book - Day Dreamers",
    "Hope Angel Fine Art",
    "Sticker - Whale",
    "Peter + June", # Brand/Artists
    "Tomorrow I‚Äôll Be Brave", # Book
    "Postcard - Oregon Coast",
    "Ashley Cuddeford",
    "Pin - Rainbow Sheep, Warrior",
    "Wander Enamel Pin",
    "Gleeful Peacock", # Brand
    "The Crowned Rabbit", # Duplicate
    "Hand Knit - Mask $", # Duplicate
    "Tarot For All Ages",
    "Undercover Corgi In Octopus", # Plush
    "Jessica Hische",
    "Single Greeting Card Elephants Pink (W)",
    "Card - I Ducking Love You",
    "Coaster Set",
    "Card - Congrats - Elephants",
    "Blanket - Numbers",
    "Voodoo Bath Bomb",
    "Mermaid Sticker",
    "Ring - Gem",
    "Cat Wilson",
    "Abcs Of Life", # Book/Theme
    "Wander Hourglass Holographic Sticker",
    "Bracelet Mala - Earthmagick",
    "Mug -Portland Or Map Icons",
    "Ring - Flower",
    "Owls", # Design/Theme
    "Note Box Elephants Dark Green (W)",
    "Hiroshige Blossom Fans - Giclee Print",
    "Clever Idiots Cat Paw Chair Socks - Tabby Cat Grey",
    "Corgi With Little Bouquet Vinyl Sticker",
    "Mushroom House", # Design/Theme
    "Ki-Shirt (White)",
    "My Neighbor Totoro Bento Lunch Box (21.98Oz, 650Ml)",
    "Bookmark - Poppy Spider",
    "Postcard Set - Portland Icon",
    "Undercover Corgi", # Plush/Theme
    "X10", # Identifier
    "Undercover Panda In Red Panda", # Plush
    "Chris Castor",
    "Don‚Äôt Be A Shit", # Slogan/Theme
    "Card - Bitches Unite",
    "Notepad - Honey Do List",
    "Blanket - Alphabet",
    "Beaverton Mug",
    "Card - Your Face",
    "Fancy As Fuck", # Slogan/Theme
    "Derek", # Artist/Name
    "Mystic Mondays Tarot",
    "Good Luck Sock", # Brand
    "My Neighbor Totoro Chopstick And Spoon With Case - Foraging",
    "Ginkgo Leaf Vinyl Sticker",
    "Patch - Van",
    "Sarah Beth Greene",
    "Ok Tarot: The Simple Deck For Everyone",
    "Postcard - Dog Lit Up On Shrooms",
    "Ki Shirt", # Duplicate
    "Sarah Jacoby",
    "Pin - Tanuki",
    "Large Notebook Elephant On Toilet (W)",
    "Postcard - Pdx Rainbow Bridge",
    "Astro Cat Enamel Pin (Orange)",
    "Serena Gingold Allen",
    "One Lane Road", # Brand/Theme
    "Love And Meanness", # Book/Theme
    "Mala By Valita",
    "Rc", # Unknown abbreviation, likely merch/POS item
    "Pin - Corgi",
    "Earrings - Antlers",
    "The Mushroom Tarot",
    "Sparkle Farm", # Brand
    "Scales Sticker",
    "Space Corgi Enamel Pin",
    "Leafs", # Design/Theme
    "Sticker - Snowy Owl",
    "My Friend Fear", # Book
    "Postcard - Mt Hood",
    "Little Sage Tarot",
    "Planner - Bon Appetit",
    "Filters", # Coffee/Tea filters
    "Michelle Rial",
    "Hidden Forest", # Design/Theme
    "X10 - Beluga Whales", # Identifier
    "Card - Fox Family",
    "Print - Extra Large",
    "Card - Rad Woman With Skates",
    "There Is Poetry In Me", # Theme/Book
    "Bookmark - Luna Bouquet",
    "Small - Adventures With Barefoot Critters", # Book size/version
    "Card - Anatomy Of A Rad Woman Yoga + Plants",
    "Deb", # Name/Brand?
    "Corgi Purse",
    "Sticker Pack",
    "Love Nikki", # Game/Theme
    "The Unipiper Cycles Through Portland", # Book
    "Card - You Are Brave",
    "Notepad",
    "Wooden Greeting Card - Great Horned Owl",
    "Spa Day", # Theme/Set
    "Pin - Penguin",
    "Catching Stars", # Theme/Design
    "Box Set",
    "Pin - Zen Cow",
    "Corgi & Puppies Vinyl Sticker",
    "J.P. Sullivan",
    "X10 Print", # Identifier
    "Ok Tarot", # Duplicate
    "#Rona 2020 Sticker",
    "Tea Towel - Bees",
    "Patch - To The Trees",
    "My Neighbor Totoro Bamboo Chopstick - Leaves",
    "Meera Lee Patel",
    "Postcard - Portland Bridge",
    "Boob Ring - Rose Gold",
    "Cranes", # Design/Theme
    "Iii: Arrows Notebook",
    "Postcard- Mount Rainier",
    "Take Miaaaaway", # POS instruction?
    "Boob Ring",
    "Tea Towel - Portland",
    "Zipper Hood", # Hoodie
    "Notebook - 100 Dot Grid Pages",
    "Yogi", # Abstract/Theme, could be tea brand but listed separately
    "Michele Maule",
    "Illuminated Journal",
    "Card - Honey Bees",
    "X10 - Oregon Coast", # Identifier
    "Pin - Van",
    "Gudetama Utensil Set (Sunny-Side Up)",
    "The Cocktail Box Co", # Kit is merch
    "Bracelet",
    "Written In The Stars Enamel Pin",
    "Wooden Greeting Card - Lovers",
    "Cambro", # Food service container brand
    "The Imagination Spot", # Duplicate
    "Multifolia", # Botanical theme/Brand
    "Rockpool", # Theme/Design
    "Wooden Greeting Card - Luna Bouquet",
    "Logo Sticker",
    "Written In The Stars Holographic Sticker",
    "Love Is Love - Set Of Pencils",
    "Stitch And Stone", # Brand
    "Card - Anatomy Of A Rad Woman Derby",
    "Bookmark - Heart Tree",
    "X10 - Giraffe Family", # Identifier
    "Card - Plenty Of Fish",
    "Christina Pierce", # Duplicate
    "Sticker - Sea Turtle",
    "If You Come To Earth", # Book
    "Fabulous Unicorn Holographic Sticker",
    "Picnic On The Mushroom", # Design/Theme
    "Card - Rad Woman With Baby",
    "Terry Cuddeford",
    "Adventures With Barefoot Critters", # Book
    "Card - Hello From The Otter Side",
    "Card - Hot Air Balloon",
    "Le Chat Noir Earings", # Typo: Earrings
    "Book - Dream Animals",
    "Postcard - Mount Hood", # Duplicate
    "Card - Shooting Star",
    "Book - The Wonderful Things You Will Be",
    "Card - Congrats - Bears",
    "Card - You Are Strong",
    "Conch Shell Earrings",
    "Things Are Shockingly Possible", # Book/Theme
    "X14 - Spiritual Unity", # Identifier
    "The Tarot Coloring Book", # Duplicate
    "Tarot De Marseille",
    "Ariel Kusby",
    "Pin - Logo",
    "Sticker - Red Mermaid",
    "Jewelery", # Typo: Jewelry
    "Postcard - Happy Holidays Hot Chocolate House",
    "Postcard - Portland Unicorns",
    "Pink Himalayan Grinder", # The grinder itself
    "Portland Round Tray",
    "Time To Wander Journal",
    "Portland 1 To 10", # Book
    "Pin - Black Sheep, Warrior",
    "Mug - Great Women Of Science",
    "Insects", # Design/Theme
    "Spirited Away Bamboo Chopstick - No-Face",
    "The Paws", # Design/Theme
    "Cocktail Box Co", # Duplicate
    "Becky Vasquez",
    "Pen",
    "Spring Break", # Theme/Event
    "Written In The Stars Journal",
    "Notebook - Hedgehogs",
    "Babies", # Theme/Section
    "Planner - Skys The Limit",
    "Katrina Liu",
    "Harvest Moon (Hoodie)",
    "Le Tarot Astrologique",
    "Tea Towel - Camping",
    "Sleeping Squirrel", # Design
    "Kimono Fans - Giclee Print",
    "Skc (Susie Chriswisser)",
    "Atom Earrings",
    "Ginkgo Leaf - Light And Dark Green Enamel Earrings",
    "Card - You Slay",
    "Deborah Underwood",
    "Wooden Greeting Card - Birthday Wishes",
    "Kiki'S Delivery Service Die Cut Lunch Bag - Jiji",
    "Postcard - Pdx Stag",
    "A Cats Guide To Money", # Book
    "Nickle Art Studio", # Duplicate
    "Sylvia Draws",
    "Who Hoo Are You?", # Book
    "Boob Necklace",
    "Revitalizing Oil - 8 Oz", # Body care likely
    "Bracelet Mala - Valita",
    "Space Corgi Holographic Sticker",
    "Merry Christmas", # Card/Theme
    "Card - Hey Boo",
    "Musical Notes Earings", # Typo: Earrings
    "Ellen Jackson",
    "Bookmark - Cornucopia",
    "Bamboo Wood Sticker",
    "Card - Keep Portland Weird - Straight Brunette",
    "Bookmark - Stag Beetle",
    "The Tall Trees Of Portland", # Book
    "Portland Denim Tote",
    "Swedish Dishcloth - Fox",
    "Swedish Dishcloth - Portland Biker Guy",
    "Card - Girl Gang",
    "Adventure Awaits", # Theme/Journal
    "Read Em And Weep Tote",
    "Calligraphy Flower Stretched Earrings",
    "Undercover Panda", # Plush/Theme
    "Kiki'S Delivery Service Round Bento Lunch Box 16.91Oz,",
    "Postcard - Cascadia",
    "Card - Outdoor Bath",
    "Single Itty Bitty Studs", # Earrings
    "X10 - Giraffe Couple", # Identifier
    "Card - Two Men Bath",
    "X14 - Many Crossings In Bridge Town", # Identifier
    "Card - Made For Each Other",
    "Card - Man And Woman In Bath",
    "Too Magical", # Theme/Slogan
    "Raindrop Circle Earrings",
    "Card - Keep Portland Weird - Fair, Blonde",
    "Paper Cup", # Supply
    "Planet Bunnie", # Brand/Artist
    "The Triangular", # Design/Shape
    "Kanji Symbol (Love) Earrings",
    "Kawaii Rainbow Enamel Pin - Glitter Edition",
    "Family Conversation Cards",
    "Samiramay Tarot",
    "Round Sustainable Fairtrade Handmade Fruit Basket Bowl",
    "Bubble Earrings",
    "Sunrise Blossoms Earrings",
    "Paper Greeting Card", # Duplicate
    "Dream World Matching Game",
    "Julie", # Artist/Name
    "Undercover Kitty", # Plush/Theme
    "My Neighbor Totoro Die Cut Lunch Bag - Blue",
    "Tits - Sticker",
    "Corgi White Eco-Friendly Tote Bag",
    "Little Little Art Co",
    "Honey Bee Tote Bag",
    "Pencil Case",
    "Illuminated", # Related to Tarot/Journal
    "My Neighbor Totoro Die Cut Lunch Bag - White",
    "Greek Bamboo Earrings",
    "Kami Mcbride",
    "Rebecca Green",
    "Swirl Earrings",
    "Fountain Pyramid Earrings",
    "Circle Array Earrings",
    "Shattered Triangle Earrings",
    "Orchid Earrings",
    "Large Flower Earrings",
    "Long Flower Earrings",
    "No Mans Land", # Theme/Design
    "Embraced Bamboo Earrings",
    "Jessica E. Pierce",
    "Extra Cup", # Supply/Charge
    "Bill Alsup",
    "Jon Klassen",
    "X10 - Rad Woman With Plants", # Identifier
    "Emily Winfield Martin", # Duplicate
    "Card - Santa Ho Ho Ho",
    "Portland Farmers Market Cookbook",
    "X10 - Silk Moth", # Identifier
    "Papa Bear", # Design/Theme
    "Card Bundle", # Duplicate
    "Dance In The Forest", # Design/Theme
    "Postcard - Pdx Unicorn",
    "Ki Tote",
    "Card - Keep Portland Weird - Curly Brunette",
    "X10 - Snowy Owl", # Identifier
    "Beehouse Dripper",
    "Penguins", # Design/Theme
    "X10 - Poppies And Butterflies", # Identifier
    "Finding Grace", # Book/Theme
    "Kitty In Boat", # Design
    "Card - Women In Bath",
    "Wooden Greeting Card - Stag Beetle",
    "Sticker - Pastel Mermaid",
    "Card - Drunk Penguin Happy Holidays",
    "Masks $10", # Duplicate
    "Notepad - Van",
    "Card - He Was A Prick",
    "X10 - Wolf", # Identifier
    "Dancing Fox", # Design
    "Greetings From Portland", # Card/Postcard
    "Brew-In-Mug Infuser",
    "Necklace Mala - Earthmagick",
    "X10 - Giraffe Mother And Baby", # Identifier
    "X10 - Fuck Yeah Fox", # Identifier
    "Tea Towel - Floral",
    "Bracelets", # Duplicate
    "Card - Capybara Holiday",
    "Bookmark - Flora & Fauna",
    "Sticker - Blue Mermaid",
    "Box Set - Reindeer",
    "Card - Be Lazy",
    "Card - Keep Portland Weird - Dark, Brunette",
    "Sticker - Take Me To The Trees",
    "Ring", # Duplicate
    "Mug - Portland Biker Guy",
    "Wooden Greeting Card - Cascadian Bouquet",
    "Hair Ties",
    "Pin - Elephant",
    "Troy", # Brand/Name
    "Julie Thomas",
    "Bookmark - Rampant Unicorn",
    "Post Cards: Old Beaverton", # Typo: Postcards
    "Stone Lantern #3 - Crystal Bead Earrings",
    "Keychain",
    "Noteworthy P&P", # Duplicate
    "Card - Miss Your Face Wreath",
    "Cubist Profile - Sapphire Bead - Giclee Print - Domed",
    "Hour", # Service/Charge
    "Patch - Compass",
    "Bodle Frog", # Brand/Design
    "Shine", # Theme/Brand
    "X10 - Whale", # Identifier
    "Creature Cups", # Mug brand
    "The Plant Room", # Theme/Brand
    "Boob Ring - Silver",
    "Notecard Set", # Duplicate
    "Elyse Breanne Design",
    "Hans Ramos",
    "Mistletoe", # Theme/Design
    "Notebook - Fox And Squirrels",
    "Bracelet Mala", # Duplicate
    "Plantsnsht", # Brand
    "Prints 11X", # Identifier
    "Ray", # Duplicate
    "X10 - Pnw Vibes", # Identifier
    "Paper Greeting Card - Good Luck",
    "Bruce", # Duplicate
    "Postcard - Burnside Bridge",
    "Box", # Container/Packaging
    "Tea Hot Pot", # Serving ware
    "Wooden Greeting Card - Birds",
    "You Are Your Best Friend", # Book/Theme
    "Washi Tape",
    "Sophia Blackall",
    "Tanuki Sticker", # Duplicate
    "Eliot", # Duplicate
    "Box Set - Owl",
    "Victoria'S Art",
    "Folding Handle Infuser",
    "Seek & Swoon", # Blanket brand likely
    "Tote Bag - Honey Bee", # Duplicate
    "Big Castle Cross", # Design
    "Hidden Cafe", # Theme/Location?
    "Booking - Hourly", # Service/Fee
    "Delivery Charge", # Service/Fee
    "Bar", # Service/POS item?
    "What Is A Woman", # Book
    "Valita", # Duplicate
    "Coaster",
    "Scoop", # Tool
    "Card Processing", # Service/Fee
    "Tea Canisters",
    "Bruce Rash", # Duplicate
    "House", # Vague, likely non-food
    "Grey Fiction", # Brand/Theme
    "X10 - Alpaca", # Identifier
    "Whale Greeting Card",
    "Filter", # Duplicate
    "Fans", # Physical fans
    "Vegan Leather Tote",
    "Wild", # Vague, likely non-food
    "Fairy Houses", # Theme/Design
    "Notebooks", # Duplicate
    "Portland Single Wine Tote",
    "Ray Wargo",
    "Corinna Luyken",
    "Pin - Narwhal",
    "Little Little Shirt",
    "Fairy Meditation", # Theme/Book
    "Card - Two Lady Bath",
    "Boho Gal Jewelry",
    "Card - Multnomah Falls",
    "Bookmark - Roots",
    "Paper Greeting Card - Follow Your Bliss",
    "No Love Like It", # Theme/Book
    "X10 - Party Of 9 Dogs", # Identifier
    "Mama Bear", # Design/Theme
    "X12 - Poppies And Butterflies", # Identifier
    "Koinobori - Children Of The Family Bookmark",
    "Space Corgi Pin", # Duplicate
    "Pointed Drop Bamboo Earrings",
    "Card - Keep Portland Weird - Straight Pink",
    "Cubist Guitar - Slate Blue Bead",
    "Blanket - New Alphabet",
    "Sticker Pack - Cows",
    "Mothers Day - Ducks", # Card?
    "Black Towel",
    "Coral Earrings",
    "Card - He Was A Prick Anyway", # Duplicate
    "Loving Embrace Earrings",
    "Circle Bamboo Earrings",
    "X10 - Diversity Children Print", # Identifier
    "Postcard - Pdx Rose",
    "Card - Happy Holidays Rabbit",
    "Card - Portland",
    "Aphantasia Sticker",
    "X10 - Bear & Hedgehog", # Identifier
    "X10 - Pufferfish", # Identifier
    "Town Square", # Location/Theme
    "X11 - Spiritual Unity", # Identifier
    "Daruma Thank You Greeting Card",
    "Sibley Backyard Birding Flashcards",
    "Hey Boo - Letterpress Card", # Duplicate
    "Cool Story Pencils",
    "Split Leaf Philodendron Leaf Blossoms Earrings",
    "Byzantine Road", # Theme/Design
    "Peace Out Card",
    "Swaddle",
    "Tribal Sun Bamboo Necklace",
    "Hello Gorgeous Pencil Set",
    "The Great Wave Bookmark",
    "This Bag Holds My Shit Together", # Tote slogan
    "Leo Vs. Draco", # Design/Theme
    "Pride Earrings",
    "Nine Lives Bookmark",
    "Mug - Stay Wild Mountains & Flowers",
    "X10 - Double Waterfall Landscape", # Identifier
    "X10 - Group Shot", # Identifier
    "Japanese Flag Bracelet",
    "Vertical Blossoms Earrings",
    "I Am Holding You In My Heart Card",
    "Corgi Express Train Enamel Pin",
    "Pin - Dolphin",
    "Clean The Dishes Sea Dragon Swedish Dishcloth",
    "Smells Of Fall", # Candle/Scent
    "Plant Lover Sticker",
    "Wooden Greeting Card - Woodland Heart",
    "Earrings - Butterflies", # Duplicate
    "Daily Plan Notepad",
    "Whale Enamel Pin",
    "Teapot",
    "Pride Bracelet",
    "Wooden Greeting Card - Christmas Tree",
    "Musubi Made For Each Other Greeting Card",
    "Alexander Hamilton Bookmark",
    "Paper Greeting Card - Reindeer",
    "Kiss Me Under The Mistletoe", # Card/Theme
    "Corgi Swaddle",
    "Card - Happy Holidays Bear",
    "Set Of", # Incomplete, likely merch
    "Box Set - Bunnies",
    "Postcard - City Of Roses",
    "Drunken Tanuki", # Design
    "Denim Portland Wine Tote",
    "Layered Teardrop Earrings",
    "X10 - Cat And Hummingbird", # Identifier
    "Card - Happy Rain Cloud",
    "X10 - Polar Bears", # Identifier
    "Pin - Yoga Sheep, Tree Pose",
    "Card - Keep Portland Weird - Med, Brunette",
    "Ring | Green",
    "Tanuki Pin", # Duplicate
    "Rorschach Ink Design Earrings",
    "On Top Of The Island", # Design/Theme
    "X14 - World Unity", # Identifier
    "Card - Laughing Reindeer",
    "Modern Floral Daily Plan Notepad",
    "Swedish Dishcloth - Sea Dragon",
    "X10 - Koala And Cub", # Identifier
    "Black Classic Blanket",
    "Winter Break", # Event/Theme
    "Calendar - 20",
    "Birthday Party", # Service/Fee
    "Card - This Too Shall Pass",
    "Box Set - Winter Hare",
    "Soapstone Cat And Mouse Set Natural White (W)",
    "Postcard - Seattle Emerald City",
    "Bookmark - Beetle",
    "Sakura Ornaments Greeting Card",
    "Muusse Design",
    "Interlocked Triangle", # Design
    "Raindrop Splashes Earrings",
    "Ohm Earrings",
    "Tote Bag", # Duplicate
    "Card - Congrats - Dogs",
    "Aegean Village", # Design/Theme
    "Marble Archway", # Design/Theme
    "Portland Animal Icon Towel",
    "Postcard - Happy Holidays"
]

food = [
    "Loose-Leaf Tea",
    "Flavor",
    "Hot Cocoa",
    "Egg & Cheddar",
    "Scone",
    "Lemonade",
    "Red Bull",
    "Griffith Street", # Assuming named food item
    "Iced Tea",
    "Flavored Iced Tea",
    "Egg Meat Cheese",
    "Little Little Loaf",
    "Build Your Own", # Custom food item
    "Kombucha",
    "Tap", # Drink from tap
    "Special-Tea", # Tea drink
    "Day Old", # Baked goods
    "Matcha Latte",
    "Egg Hummus Pesto",
    "Tea Toddy",
    "Bag O' Beans", # Coffee beans
    "Tap - Kombucha",
    "Brew Dr", # Kombucha brand
    "Awkward Aardvark", # Named food item
    "Arnold Palmer",
    "Kind", # Kind bar
    "Golden Fire", # Named drink/tea
    "Earl Grey",
    "Ex. Flavor", # Extra flavor
    "Domo Froppo", # Assuming named food/drink item
    "Goblin King", # Assuming named food/drink item
    "Jasmine Pearls",
    "Special Tea", # Duplicate?
    "Kyushu Sencha",
    "Fruit",
    "Honey Lemon Ginger",
    "Tom Swanson", # Assuming named food item
    "Esp Float", # Espresso Float
    "Cranberry Sencha",
    "Cold-Pressed Lemonade",
    "Pure", # Drink/Tea type
    "Big Train Chai",
    "African Grey", # Assuming tea/coffee
    "Caveman", # Assuming named food item
    "House Iced Tea",
    "Chaga",
    "Vegan Griffith Street",
    "Hot Chocolate",
    "Welchs",
    "Shake",
    "Egusto", # Assuming named food item
    "Carton", # Milk/Juice likely
    "Green Jade", # Tea
    "Drip", # Coffee
    "Bee Local Honey",
    "Vegan Awkward Aardvark",
    "Hojicha",
    "Genmaicha",
    "Egusto Swanson",
    "Scone - Blueberry",
    "Annie'S", # Snack brand likely
    "The Dook", # Assuming named food item
    "Vive", # Wellness shot brand
    "Sweet Iced Tea",
    "Tea",
    "Donut",
    "Puerh",
    "Tlc", # Assuming tea/drink name
    "Gum",
    "Zone Perfect", # Nutrition bar brand
    "Jacobsen", # Salt brand
    "London Fog",
    "Egusto Mato",
    "Mint/Gum",
    "Mint",
    "Jacobsen Salt Co",
    "Apple",
    "Sweet Tea", # Duplicate
    "Milk Tea",
    "Redbull", # Duplicate
    "Golden Yogi", # Named drink/tea
    "Black Butte", # Beer brand likely
    "Kure Bar", # Food bar brand likely
    "Pumpkin Pie",
    "Hibiscus",
    "Weekly Special", # Assuming food special
    "English Breakfast - Black Tea",
    "Oatly", # Oat milk brand
    "Chamomile",
    "Kyushu", # Tea related
    "Nonmeat Garden Home", # Veggie item
    "Fresh Lemonade",
    "Jasmine Pearl", # Duplicate
    "Rockstar", # Energy drink
    "Redbush Chai",
    "Black Garlic Salt",
    "Day Olds", # Duplicate
    "White Peony",
    "Ninkasi", # Beer brand
    "Orange",
    "Nonmeat Careless Whisper", # Veggie item
    "Rooibos",
    "Smith Tea", # Tea brand
    "Lavender",
    "Concord", # Grape flavor/juice likely
    "Co2 Brew", # Coffee brew method
    "Inji", # Ginger related? Assume food
    "Feel Better", # Tea related
    "Rosemary Salt",
    "Lavander Rose - White Tea Blend", # Typo: Lavender
    "Raw Bee Pollen",
    "Bachan‚Äôs Japanese Barbecue Sauce",
    "Made Good", # Snack brand likely
    "Feel Better - Herbal Tea",
    "Super Joy - Uganda", # Coffee beans likely
    "Booberry", # Cereal/flavor likely
    "Lemon Zest Salt",
    "Black Lava Salt",
    "Almonds",
    "Yogurt",
    "Galway Girl", # Assuming named food/drink item
    "Steven Smith", # Duplicate tea brand
    "Meat Garden Home",
    "Haiku Peach - White Tea Blend",
    "Burn Brew", # Coffee/Tea name likely
    "Proud Source", # Water brand likely
    "Jasmine Pearl - Green Tea", # Duplicate
    "Meat Careless Whisper",
    "Puerh Queen", # Tea name likely
    "Karo", # Corn syrup? Ingredient
    "Finca Kilamanjaro - El Salvador", # Coffee beans
    "Chocolate",
    "Cheesecake",
    "Clif", # Clif bar brand
    "Earl Grey - Black Tea", # Duplicate
    "Honey Cup - Herbal Tea",
    "Catering", # Food Service
    "Red Alaea Salt",
    "Honest Tea", # Tea brand
    "Naked", # Juice brand likely
    "Hot Pot", # Meal type?
    "Ice Cream",
    "Cafe", # Coffee/drink charge?
    "Altoide", # Mint brand likely (Typo?)
    "Honey Cup", # Duplicate
    "Green Jade - Oolong", # Duplicate
    "Widmer", # Beer brand likely
    "Buoy", # Beer brand likely
    "Injy", # Duplicate
    "Guatemala El Injerto", # Coffee beans
    "Super Joy - China", # Coffee beans likely
    "Super Joy - Ethiopia", # Coffee beans likely
    "Fruit Platter",
    "Ginger", # Flavor/Ingredient
    "Double", # Drink modifier?
    "Espresso Embassy", # Coffee place/beans
    "Super Joy - Yunnan", # Coffee beans likely
    "Loose Leaf Tea", # Duplicate
    "Ruby Nectar - Herbal Tea",
    "Hakutsuru - Nigori", # Sake
    "Kikusui Shuzo", # Sake brand
    "Hakutsuru - Nigori - Rich & Sweet", # Sake
    "Vegan Dining Month", # Event/Special
    "Golden Fire - Herbal Tea", # Duplicate
    "Peru Monteverde" # Coffee beans
]

rare_items = df_uncleaned['item_name'].value_counts().to_frame('counts').query('counts < 10').index.tolist()
drink_items = df_uncleaned.query('dish_category.isin(@drink_categories)')['item_name'].value_counts().index.tolist()
items_to_remove = merch + drink_items + rare_items

df_uncleaned = remove_numbers(df_uncleaned, 'item_name')

drink_categories = ["Coffee & Tea","Dairy Drink","Alcohol","Soda","Water","Juice","Sports & Health Drink"]

# Step 1: Apply initial modifications
df_intermediate = fully_relabel_and_consolidate(
    df_uncleaned,
    remove=items_to_remove, # Apply initial removals
    name_changes=dish_names, # Apply general name changes
    modification_name_changes=modification_rules_step1, # Apply Step 1 modification rules
    # Pass all lists for initial relabeling/recategorization
    vegan_list=vegan,
    vegetarian_list=vegetarian,
    meat_list=meat,
    alcohol_list=alcoholic_drinks, # Corrected parameter name
    drinks_list=non_alcoholic_drinks, # Corrected parameter name
    merch=merch,
    rare=rare,
    unknown=unknown,
    remove_categories=drink_categories
)

# Step 2: Apply final modifications and set final flags/categories
df_relabeled = fully_relabel_and_consolidate(
    df_intermediate, # Use the output of Step 1 as input
    # No removals or general name changes needed again
    modification_name_changes=modification_rules_step2, # Apply Step 2 modification rules
    # Pass lists again for final relabeling/recategorization based on updated names
    vegan_list=vegan,
    vegetarian_list=vegetarian,
    meat_list=meat,
    alcohol_list=alcoholic_drinks, # Corrected parameter name
    drinks_list=non_alcoholic_drinks, # Corrected parameter name
    merch=merch,
    rare=rare,
    unknown=unknown
)

df_relabeled.to_parquet(f"data/4_palate_data_parquet_relabeled/relabeled/{loc_id}_sales_and_menu.parquet")
consolidating_names = {}
for dish, _, _ in DISH_CONFIG:
    consolidating_names[dish] = ["Vegan " + dish, "Vegetarian " + dish, "Meat " + dish]

df_consolidated = df_relabeled.pipe(rename_items, name_changes = consolidating_names)

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, top_n=30)

df_consolidated.to_parquet(f"data/4_palate_data_parquet_relabeled/consolidated/{loc_id}_sales_and_menu.parquet")

In [ ]:
print(df_consolidated['item_name'].value_counts().to_string())

In [ ]:
# 'item_modifications.str.contains("Vegan")'
# bacon_conditions = ['item_modifications.str.contains("Bacon")',
#                     'item_name.str.contains("Bacon")'
#                     'item_name.str.contains("Anam A Nom")',
#                     'item_name.str.contains("Yeti Sandwich")',
#                     'item_name.str.contains("Fullmetal Alchemist")',
#                     'item_name.str.contains("The Jesse Sandwich")',]
# df.query(' or '.join(bacon_conditions)).head()

In [ ]:
"Item :" + df[['item_name','item_modifications']].value_counts().reset_index()['item_name'] + "; Modifications: " + df[['item_name','item_modifications']].value_counts().reset_index()['item_modifications']

In [ ]:
vegan_str = before_after_details_true.loc[loc_id,'promo_name'][0]
bacon_str = before_after_details_true.loc[loc_id,'promo_name'][1]
promo_item_containing = df_uncleaned.loc[lambda df: (df['item_name'].str.contains(vegan_str, na=False) & 
                                                    df['item_name'].str.contains(bacon_str, na=False) | 
                                                    df['item_modifications'].str.contains(vegan_str, na=False) &
                                                    df['item_modifications'].str.contains(bacon_str, na=False)) &
                                         ~df['item_modifications'].str.contains("Turkey Bacon|Real Bacon|No Bacon|Normal Bacon|Regular Bacon|\+Bacon", na=False) |
                                         df['item_modifications'].str.contains("Bakon|Thrilling")]
promo_item_containing_2 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Make It Vegan", na=False)]
promo_item_containing_3 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan Egg|Just Egg", na=False)]
promo_item_containing_4 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan Cream Cheese", na=False)]
promo_item_containing_5 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan", na=False) | df['item_name'].str.contains("Vegan")]
promo_item_containing_6 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan Sausage", na=False)]
promo_item_containing_7 = df_uncleaned.loc[lambda df: df['item_modifications'].str.contains("Vegan Cheese|Vegan Cheddar", na=False)]
awkward_aardvark = df_uncleaned.loc[lambda df: df['item_name'].str.contains("Awkward Aardvark", na=False)]
print(promo_item_containing['item_name'].unique().tolist())

plt.plot(promo_item_containing.resample('W')['item_quantity'].sum())
plt.plot(promo_item_containing_2.resample('W')['item_quantity'].sum())
plt.plot(promo_item_containing_3.resample('W')['item_quantity'].sum())
plt.plot(promo_item_containing_4.resample('W')['item_quantity'].sum())
#plt.plot(promo_item_containing_5.resample('W')['item_quantity'].sum())
plt.plot(promo_item_containing_6.resample('W')['item_quantity'].sum())
plt.plot(promo_item_containing_7.resample('W')['item_quantity'].sum())

#plt.plot(awkward_aardvark.resample('W')['item_quantity'].sum())
#plt.plot(df_uncleaned.resample('W')['item_quantity'].sum())
plt.axvline(x=before_after_details_true.loc[loc_id, 'cross_over_date'], color='red', linestyle='--', label='Promo Date')
plt.title("Plant-Based Analog")
#Legend
plt.legend([
    'Vegan Bacon',
    'Make It Vegan',
    'Vegan Egg',
    'Vegan Cream Cheese',
    #'Vegan',
    'Vegan Sausage',
    'Vegan Cheese'
])
plt.xticks(rotation=50)
plt.show()

In [ ]:
dish_conditions_egg_meat = [df['item_name'].str.contains(dish) for dish in ['Egg Meat', 'Egg Meat Cheese', 'Ex Meat']]
modification_meat_conditions = [df['item_modifications'].str.contains(meat) for meat in ['Sausage', 'Bacon', 'Ham']]
print(df
      .loc[reduce(np.logical_or, dish_conditions_egg_meat) & ~reduce(np.logical_or, modification_meat_conditions)]
      .value_counts(['item_name','item_modifications'], sort=False)
      .to_string())

In [ ]:
plot_dish_time_series(df_cleaned, loc_id, before_after_details_true)